# Deteccion Sitios Web Fraudulentos

In [14]:
import requests
import tldextract
import whois
import random
import re
from fake_useragent import UserAgent
from faker import Faker
from datetime import datetime
from tqdm.notebook import tqdm
import pandas as pd
from io import StringIO

# Instancias globales
fake = Faker()
ua = UserAgent()

### Ingesta de Datos Inicial

Inicialmente, se plantea realizar un analisis de phishing las caractersiticas tecnicas del dominio y patrones de la url:

In [16]:
def obtener_datos_sitio(url):
    """
    Obtiene información sobre un sitio web real:
    Extrae información básica del dominio (TLD, longitud, caracteres especiales).
    Verifica si el sitio usa HTTPS y tiene SSL.
    Obtiene datos de WHOIS (registrador, país, antigüedad del dominio).
    Asigna la etiqueta is_phishing = 0 porque asumimos que los sitios extraídos son legítimos.
    """
    datos = {"domain": url}

    # Extraer información del dominio
    ext = tldextract.extract(url)
    datos["tld"] = f"{ext.suffix}"  # Extrae el TLD (ej: .com.ar, .gob.ar)

    # Verificar si usa HTTPS
    if url.startswith("https"):
        datos["is_https"] = 1
    else:
        datos["is_https"] = 0

    # Longitud de la URL
    datos["url_length"] = len(url)

    # Contar caracteres especiales
    datos["num_hyphens"] = url.count("-")
    datos["num_digits"] = sum(c.isdigit() for c in url)
    datos["num_special_chars"] = len(re.findall(r"[@#?$%^&*]", url))
    datos["num_path_segments"] = url.count("/")

    # Intentar obtener el certificado SSL
    try:
        response = requests.get(f"https://{url}", headers={"User-Agent": ua.random}, timeout=5)
        if response.status_code == 200:
            datos["has_ssl"] = 1
        else:
            datos["has_ssl"] = 0
    except requests.exceptions.RequestException:
        datos["has_ssl"] = 0

    # Obtener información WHOIS
    try:
        whois_info = whois.whois(url)
        datos["whois_registrar"] = whois_info.registrar if whois_info.registrar else "Desconocido"
        datos["whois_country"] = whois_info.country if whois_info.country else "Desconocido"
        if whois_info.creation_date:
            if isinstance(whois_info.creation_date, list):
                creation_date = whois_info.creation_date[0]
            else:
                creation_date = whois_info.creation_date
            datos["domain_age_days"] = (datetime.now() - creation_date).days
        else:
            datos["domain_age_days"] = -1
    except:
        datos["whois_registrar"] = "Error"
        datos["whois_country"] = "Error"
        datos["domain_age_days"] = -1

    # Asumimos que todos los sitios reales son legítimos
    datos["is_phishing"] = 0

    return datos


Se opto por emplear el sitio web [*Tranco*](https://tranco-list.eu/) que rankea los sitios web con mayor trafico y provee una base de datos actualizada diaramente

In [ ]:
# URL of the CSV file to download
csv_url = "https://tranco-list.eu/download/X45KN/1000000"
argentina_web_sites_file = "sitios_argentinos.csv"
dataset_output_file = "dataset_sitios_argentinos.csv"

In [ ]:
# Descargar el archivo CSV
response = requests.get(csv_url)
response.raise_for_status()

# Parsear el contenido CSV
csv_content = StringIO(response.text)

# Crear dataframe y añadir encabezados al CSV
headers = ["rank_position", "url"]
df = pd.read_csv(csv_content, header=None, names=headers)

# Quitar la primer columna que es un índice
df = df.iloc[:, 1:]

print(df.head())

# Guardar como archivo CSV
df.to_csv(argentina_web_sites_file, index=False)

print(f"Arhivo descargado y guardado como: '{argentina_web_sites_file}'")



Crear dataset a partir del la listia de sitios obtenida

In [ ]:
NOMBRE_COLUMNA_URLS = headers[1]
LIMIT_ROWS = 100

df_input = pd.read_csv(argentina_web_sites_file)

# Convertir la columna de dominios en una lista
sitios_a_procesar = df_input[NOMBRE_COLUMNA_URLS].dropna().tolist()

datos_reales = []

# Usamos tqdm para una barra de progreso, ideal para listas largas.
# El proceso puede tardar varios minutos (o más) para 3200 registros.
for sitio in tqdm(sitios_a_procesar, desc="Extrayendo datos de sitios web"):
    try:
        datos_sitio = obtener_datos_sitio(sitio)
        datos_reales.append(datos_sitio)
    except Exception as e:
        print(f"❌ Error irrecuperable con {sitio}: {e}. Saltando al siguiente.")
        # Opcional: registrar los errores en la lista también
        # datos_reales.append({'domain': sitio, 'error': str(e)})

# Convertir la lista de resultados en un DataFrame
df_reales = pd.DataFrame(datos_reales)

# Mostrar las primeras filas y la forma del DataFrame resultante
print("\n✅ Proceso completado.")
print(f"Se procesaron {len(df_reales)} registros.")
print("\nPrimeras 5 filas del DataFrame resultante:")
print(df_reales.head())

df_reales.to_csv(dataset_output_file, index=False)
print(f"\nResultados guardados exitosamente en: '{dataset_output_file}'")


Extrayendo datos de sitios web:   0%|          | 0/1000000 [00:00<?, ?it/s]

2025-06-25 22:39:33,043 - whois.whois - ERROR - Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
2025-06-25 22:41:37,793 - whois.whois - ERROR - Error trying to connect to socket: closing socket - timed out
2025-06-25 22:44:44,199 - whois.whois - ERROR - Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed


### Preprocesamiento Estructural

### Analisis Exploratorio de Datos (EDA)

### Scraping Complementario Inicial

### Visualizacion